In [ ]:
# -*- coding: utf-8 -*-
import os
import json
import time
import numpy as np
import tensorflow as tf

# ==========================================
# KONFIGURASI PATH
# ==========================================
# Arahkan ke file TFLite asli MCU-Quake
TFLITE_PATH = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/mcquake_ori_file/Code & Figure demo/Pre-trained model/MCU-Quake 5-20/lite_model.tflite'
# Path ke direktori embedding KDE Indonesia Anda
# PENTING: harus SAMA PERSIS dengan EMB_DIR di sel export C-array, supaya
# ringkasan kelayakan di sel ini benar-benar mencerminkan data yang dideploy.
EMB_DIR = "/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/retraining_mcu_q_indonesia/output_eval_03"
EMBEDDING_DIM = 32

def main():
    print("="*60)
    print("BUILD & PROFILING: MCU-QUAKE (KOMPONEN Z) DENGAN EMBEDDING INDONESIA")
    print("="*60)

    # 1. Analisis Model TFLite Asli (Feature Extractor)
    if os.path.exists(TFLITE_PATH):
        flash_size_kb = os.path.getsize(TFLITE_PATH) / 1024.0

        interpreter = tf.lite.Interpreter(model_path=TFLITE_PATH)
        interpreter.allocate_tensors()

        tensor_details = interpreter.get_tensor_details()

        # Pisahkan tensor bobot/konstanta (sudah terhitung di flash_size_kb) dari
        # tensor aktivasi runtime, supaya "Estimasi RAM" tidak menjumlah bobot dua kali.
        # Trik: tensor konstanta sudah terisi begitu allocate_tensors() selesai (sebelum
        # invoke sama sekali dipanggil), sedangkan tensor aktivasi murni belum valid dibaca.
        activation_bytes = 0
        for t in tensor_details:
            if t['shape'] is None or len(t['shape']) == 0:
                continue
            size_bytes = int(np.prod(t['shape'])) * np.dtype(t['dtype']).itemsize
            try:
                interpreter.get_tensor(t['index'])
                # Berhasil dibaca sebelum invoke -> ini bobot/konstanta, sudah
                # terhitung di flash_size_kb, jangan didobel sebagai RAM.
            except ValueError:
                activation_bytes += size_bytes

        # Uji latensi inferensi dummy (dtype mengikuti tipe input model asli --
        # bisa float32 ATAU int8 tergantung apakah model asli sudah terkuantisasi)
        input_details = interpreter.get_input_details()
        input_shape = input_details[0]['shape']
        input_dtype = input_details[0]['dtype']
        if np.issubdtype(input_dtype, np.integer):
            dummy_input = np.random.randint(-128, 127, size=input_shape).astype(input_dtype)
        else:
            dummy_input = np.random.randn(*input_shape).astype(input_dtype)

        interpreter.set_tensor(input_details[0]['index'], dummy_input)
        start_time = time.time()
        for _ in range(100):
            interpreter.set_tensor(input_details[0]['index'], dummy_input)
            interpreter.invoke()
        latency_ms = ((time.time() - start_time) / 100) * 1000

        print(f"\n[A] Analisis Feature Extractor (TFLite):")
        print(f"    - Ukuran Flash (ROM)      : {flash_size_kb:.2f} KB")
        print(f"    - Estimasi RAM aktivasi*  : {activation_bytes / 1024:.2f} KB")
        print(f"    - Latensi Rata-rata       : {latency_ms:.2f} ms (di PC)")
        print(f"      * batas atas kasar, tanpa simulasi reuse buffer antar layer --")
        print(f"        kebutuhan arena riil TFLite Micro di MCU bisa lebih kecil.")
    else:
        print(f"\n[!] File TFLite tidak ditemukan di: {TFLITE_PATH}")
        print("    Pastikan Anda mengarahkannya ke file .tflite bawaan repositori.")

    # 2. Analisis Beban Memori KDE Indonesia (Khusus Komponen Z)
    print(f"\n[B] Analisis Ruang Probabilitas (KDE Komponen Z):")
    file_path = os.path.join(EMB_DIR, "Embedding data, Z.json")

    total_vectors = 0
    if os.path.exists(file_path):
        with open(file_path, "r") as f:
            data = json.load(f)
        n_count = len(data.get("noise", []))
        le_count = len(data.get("le", []))
        total_vectors = n_count + le_count
        print(f"    - Komponen Z: {total_vectors} vektor ({n_count} Noise, {le_count} LE)")
    else:
        print(f"    - Komponen Z: File JSON tidak ditemukan di direktori!")

    # Kalkulasi memori KDE dengan kuantisasi INT8 (1 byte per parameter) --
    # angka ini sekarang cocok dengan format yang benar-benar ditulis sel export C-array
    # (int8_t + skala, bukan float 4-byte seperti sebelumnya).
    kde_ram_kb = (total_vectors * EMBEDDING_DIM * 1) / 1024.0

    print(f"\n    > Total Vektor Laten Komponen Z : {total_vectors} titik")
    print(f"    > Beban Memori Tambahan (INT8)  : {kde_ram_kb:.2f} KB")

    print("\n" + "="*60)
    print("RINGKASAN KELAYAKAN TINYML (DEPLOYMENT SUMMARY)")
    print("="*60)
    print(f"Model dengan Komponen Z tunggal siap di-deploy ke ESP32/STM32.")
    print(f"Footprint memori sistem terintegrasi menjadi sangat ramping (~{kde_ram_kb:.2f} KB untuk KDE).")
    print("="*60)

if __name__ == "__main__":
    main()

In [ ]:
# -*- coding: utf-8 -*-
import os
import json
import numpy as np
import tensorflow as tf

# ==========================================
# KONFIGURASI PATH
# ==========================================
# Path model Keras hasil retraining
KERAS_MODEL_PATH = "/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/retraining_mcu_q_indonesia/output_models/frozen_extractor_indonesia_Z.keras"

# Path tujuan untuk model TFLite
TFLITE_OUTPUT_PATH = "/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/retraining_mcu_q_indonesia/output_models/frozen_extractor_indonesia_Z.tflite"

# Data mentah untuk kalibrasi kuantisasi INT8 (dipakai representative_dataset_gen)
REPRESENTATIVE_DATA_PATH = "/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/retraining_mcu_q_indonesia/data_indonesia/indonesia_test_data.json"
NUM_CALIBRATION_SAMPLES = 100

def make_representative_dataset_gen(input_shape):
    with open(REPRESENTATIVE_DATA_PATH, "r") as f:
        raw_data = json.load(f)

    keys = list(raw_data.keys())
    sample_count = min(NUM_CALIBRATION_SAMPLES, len(keys))

    def representative_dataset_gen():
        for k in keys[:sample_count]:
            wave = np.array(raw_data[k]["Z"][:700], dtype=np.float32)
            wave = wave.reshape([1] + list(input_shape[1:]))
            yield [wave]

    return representative_dataset_gen

def convert_keras_to_tflite():
    print("="*60)
    print("KONVERSI KERAS KE TFLITE (INT8 PENUH)")
    print("="*60)
    print(f"[INFO] Memuat model Keras dari:\n       {KERAS_MODEL_PATH}")

    if not os.path.exists(KERAS_MODEL_PATH):
        print("\n❌ Gagal: File .keras tidak ditemukan! Pastikan proses pelatihan sudah selesai.")
        return

    # 1. Memuat model Keras
    model = tf.keras.models.load_model(KERAS_MODEL_PATH, compile=False)

    # 2. Inisialisasi TFLite Converter
    converter = tf.lite.TFLiteConverter.from_keras_model(model)

    # Kuantisasi INT8 penuh (bobot + aktivasi + input/output), supaya ukuran
    # dan kebutuhan tensor arena di MCU sekecil model asli MCU-Quake (~17 KB),
    # bukan versi float32 yang ~3x lebih besar dan jauh lebih lambat di ESP32-S3.
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset = make_representative_dataset_gen(model.input_shape)
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type = tf.int8
    converter.inference_output_type = tf.int8

    # 3. Proses Konversi
    print("[INFO] Memulai proses konversi ke format TFLite (INT8)...")
    tflite_model = converter.convert()

    # 4. Menyimpan file TFLite
    with open(TFLITE_OUTPUT_PATH, "wb") as f:
        f.write(tflite_model)

    print(f"\n✅ Ekspor Berhasil!")
    print(f"   Lokasi Tersimpan : {TFLITE_OUTPUT_PATH}")
    print(f"   Ukuran TFLite    : {len(tflite_model) / 1024:.2f} KB")
    print("="*60)

if __name__ == "__main__":
    convert_keras_to_tflite()

In [ ]:
# -*- coding: utf-8 -*-
# DIAGNOSTIK: cek tensor mana yang gagal ter-kuantisasi INT8 penuh.
# Jalankan sel ini SETELAH sel konversi (di atas) selesai jalan,
# supaya TFLITE_OUTPUT_PATH sudah ada isinya.
import tensorflow as tf

interpreter = tf.lite.Interpreter(model_path=TFLITE_OUTPUT_PATH)
interpreter.allocate_tensors()

float_tensors = []
for t in interpreter.get_tensor_details():
    print(t['index'], t['name'], t['dtype'])
    if t['dtype'].__name__ == 'float32':
        float_tensors.append(t['name'])

print("\n" + "="*60)
if float_tensors:
    print(f"⚠️  {len(float_tensors)} tensor MASIH float32 (belum ter-kuantisasi):")
    for name in float_tensors:
        print("   -", name)
else:
    print("✅ Semua tensor sudah INT8 -- model terkuantisasi penuh.")
print("="*60)


In [ ]:
# -*- coding: utf-8 -*-
# Konversi MODEL ASLI Zhi Geng (dipotong di layer 32-dimensi) ke TFLite INT8.
#
# Ini model yang cocok dengan vektor KDE 32-dimensi yang sudah ada
# (output_eval_03/Embedding data, Z.json) -- sesuai skenario "kde_reembed" di
# evaluate_kde_re_embedding_all.py. BUKAN model hasil retrain Indonesia
# (frozen_extractor_indonesia_Z.keras, itu 8-dimensi & arsitektur berbeda).
import os
import json
import numpy as np
import tensorflow as tf

# Path SavedModel asli (folder ini juga berisi lite_model.tflite asli)
MODEL_PRETRAINED_PATH = "/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/mulai_juli/mcquake_ori_file/Code & Figure demo/Pre-trained model/MCU-Quake 5-20"

# Output: model 32-dimensi yang akan menggantikan mcu_quake_model.h
TFLITE_OUTPUT_PATH_ORIG32 = "/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/retraining_mcu_q_indonesia/output_models/mcu_quake_original_32d.tflite"

# Data mentah untuk kalibrasi kuantisasi INT8
REPRESENTATIVE_DATA_PATH = "/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/retraining_mcu_q_indonesia/data_indonesia/indonesia_test_data.json"
NUM_CALIBRATION_SAMPLES = 100


def truncate_at_32d(full_model):
    """Potong model di layer pertama yang output-nya persis 32 dimensi --
    logika ini identik dengan evaluate_kde_re_embedding_all.py, supaya
    embedding yang dihasilkan model ini cocok dengan vektor KDE yang sudah ada."""
    for layer in full_model.layers:
        shape = getattr(layer, "output_shape", None)
        if isinstance(shape, tuple) and shape[-1] == 32:
            return tf.keras.Model(inputs=full_model.inputs, outputs=layer.output)
    raise ValueError("Tidak ditemukan layer dengan output 32 dimensi di model ini.")


def make_representative_dataset_gen(input_shape):
    with open(REPRESENTATIVE_DATA_PATH, "r") as f:
        raw_data = json.load(f)

    keys = list(raw_data.keys())
    sample_count = min(NUM_CALIBRATION_SAMPLES, len(keys))

    def representative_dataset_gen():
        for k in keys[:sample_count]:
            wave = np.array(raw_data[k]["Z"][:700], dtype=np.float32)
            wave = wave.reshape([1] + list(input_shape[1:]))
            yield [wave]

    return representative_dataset_gen


def convert_original_32d_to_tflite():
    print("=" * 60)
    print("KONVERSI MODEL ASLI (DIPOTONG 32D) KE TFLITE INT8")
    print("=" * 60)
    print(f"[INFO] Memuat SavedModel asli dari:\n       {MODEL_PRETRAINED_PATH}")

    if not os.path.exists(MODEL_PRETRAINED_PATH):
        print(f"\n❌ Gagal: SavedModel tidak ditemukan di path di atas.")
        return

    full_model = tf.keras.models.load_model(MODEL_PRETRAINED_PATH, compile=False)
    truncated_model = truncate_at_32d(full_model)
    print(f"[INFO] Model dipotong -- output shape: {truncated_model.output_shape}")

    converter = tf.lite.TFLiteConverter.from_keras_model(truncated_model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset = make_representative_dataset_gen(truncated_model.input_shape)
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type = tf.int8
    converter.inference_output_type = tf.int8

    print("[INFO] Memulai proses konversi ke format TFLite (INT8)...")
    tflite_model = converter.convert()

    with open(TFLITE_OUTPUT_PATH_ORIG32, "wb") as f:
        f.write(tflite_model)

    print(f"\n✅ Ekspor Berhasil!")
    print(f"   Lokasi Tersimpan : {TFLITE_OUTPUT_PATH_ORIG32}")
    print(f"   Ukuran TFLite    : {len(tflite_model) / 1024:.2f} KB")
    print("=" * 60)


if __name__ == "__main__":
    convert_original_32d_to_tflite()


In [ ]:
# -*- coding: utf-8 -*-
import os
import json
import numpy as np

# ==========================================
# KONFIGURASI PATH
# ==========================================
# Model 32-dimensi (asli Zhi Geng, dipotong di layer dense_1) -- cocok dengan
# vektor KDE 32-dimensi di bawah. Diproduksi oleh sel "KONVERSI MODEL ASLI
# (DIPOTONG 32D)" di atas.
TFLITE_PATH = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/retraining_mcu_q_indonesia/output_models/mcu_quake_original_32d.tflite'

# Ubah EMB_DIR hanya sampai nama foldernya saja
# PENTING: harus SAMA PERSIS dengan EMB_DIR di sel analisis kelayakan,
# supaya ringkasan di sana benar-benar mencerminkan data yang diekspor di sini.
EMB_DIR = "/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/retraining_mcu_q_indonesia/output_eval_03"

# Direktori Output untuk file C++
OUTPUT_DIR = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/retraining_mcu_q_indonesia/output_models'

def ensure_dir():
    if not os.path.exists(OUTPUT_DIR):
        os.makedirs(OUTPUT_DIR)

def export_tflite_to_c_array():
    print("\n[1] Mengekspor Model TFLite ke C-Array...")
    if not os.path.exists(TFLITE_PATH):
        print(f"  ❌ File tidak ditemukan: {TFLITE_PATH}")
        return

    with open(TFLITE_PATH, "rb") as f:
        tflite_data = f.read()

    hex_array = [f"0x{b:02x}" for b in tflite_data]
    c_code = "#ifndef MCU_QUAKE_MODEL_H\n#define MCU_QUAKE_MODEL_H\n\n"
    c_code += "// File TFLite Feature Extractor MCU-Quake\n"
    c_code += f"const unsigned int mcu_quake_model_len = {len(tflite_data)};\n"
    c_code += "const unsigned char mcu_quake_model[] = {\n    "

    for i in range(0, len(hex_array), 12):
        c_code += ", ".join(hex_array[i:i+12]) + ",\n    "
    c_code = c_code.rstrip(",\n    ") + "\n};\n\n#endif // MCU_QUAKE_MODEL_H"

    out_path = os.path.join(OUTPUT_DIR, "mcu_quake_model.h")
    with open(out_path, "w") as f:
        f.write(c_code)
    print(f"  ✅ Tersimpan: {out_path} ({len(tflite_data)/1024:.2f} KB)")

def quantize_int8(vectors):
    """Kuantisasi simetris INT8 (per-array, skala tunggal) untuk vektor referensi KDE."""
    arr = np.array(vectors, dtype=np.float32)
    max_abs = np.max(np.abs(arr)) if arr.size > 0 else 1.0
    scale = (max_abs / 127.0) if max_abs > 0 else 1.0
    quantized = np.clip(np.round(arr / scale), -128, 127).astype(np.int8)
    return quantized, scale

def export_kde_z_to_c_array():
    print("\n[2] Mengekspor Matriks KDE Komponen Z ke C-Array (INT8)...")

    # Gabungkan folder EMB_DIR dengan nama file json-nya
    json_path = os.path.join(EMB_DIR, "Embedding data, Z.json")

    if not os.path.exists(json_path):
        print(f"  ❌ File tidak ditemukan: {json_path}")
        return

    with open(json_path, "r") as f:
        data = json.load(f)

    noise_vecs = data.get("noise", [])
    le_vecs = data.get("le", [])

    if len(noise_vecs) == 0:
        print("  ❌ Data vektor kosong.")
        return

    c_code = "#ifndef KDE_Z_VECTORS_H\n#define KDE_Z_VECTORS_H\n\n"
    c_code += "#include <stdint.h>\n\n"
    c_code += "// Ruang Probabilitas Lokal Indonesia (Komponen Z)\n"
    c_code += "// Disimpan sebagai INT8 (bukan float) -- 4x lebih hemat flash/RAM.\n"
    c_code += "// Nilai asli didapat lewat: nilai_float = kode_int8 * SCALE\n\n"

    # --- FUNGSI PEMBANTU UNTUK FORMATTING ---
    def format_array(name, vectors):
        quantized, scale = quantize_int8(vectors)
        dim = quantized.shape[1]

        code = f"const int NUM_{name.upper()} = {len(vectors)};\n"
        code += f"const float KDE_{name.upper()}_SCALE = {scale:.10f}f;\n"
        code += f"const int8_t kde_{name.lower()}_vectors[][{dim}] = {{\n"
        for row in quantized:
            formatted_vec = ", ".join(str(int(v)) for v in row)
            code += f"    {{{formatted_vec}}},\n"
        code = code.rstrip(",\n") + "\n};\n\n"
        return code

    # Menjalankan formatting untuk derau (Noise) dan gempa (LE)
    c_code += format_array("NOISE", noise_vecs)
    c_code += format_array("LE", le_vecs)
    c_code += "#endif // KDE_Z_VECTORS_H"

    out_path = os.path.join(OUTPUT_DIR, "kde_z_vectors.h")
    with open(out_path, "w") as f:
        f.write(c_code)

    print(f"  ✅ Tersimpan: {out_path} (INT8, {len(noise_vecs) + len(le_vecs)} vektor)")

def main():
    print("="*60)
    print("GENERATE C-ARRAY UNTUK ESP32-S3 DEPLOYMENT")
    print("="*60)
    ensure_dir()
    export_tflite_to_c_array()
    export_kde_z_to_c_array()
    print("="*60)
    print("Proses selesai. Pindahkan file .h di dalam folder 'esp32_deployment_files'")
    print("ke dalam folder proyek C++ / Arduino IDE Anda.")
    print("="*60)

if __name__ == "__main__":
    main()

In [ ]:
# -*- coding: utf-8 -*-
import json
import os

# ==========================================
# KONFIGURASI PATH
# ==========================================
DATA_JSON_PATH = "/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/retraining_mcu_q_indonesia/data_indonesia/indonesia_test_data.json"
OUTPUT_DIR = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/retraining_mcu_q_indonesia/data_indonesia'
OUTPUT_FILE = os.path.join(OUTPUT_DIR, "test_samples.h")

def main():
    # Memastikan folder output tersedia
    if not os.path.exists(OUTPUT_DIR):
        os.makedirs(OUTPUT_DIR)
        print(f"📁 Membuat folder baru: {OUTPUT_DIR}")

    print("Membaca dataset pengujian...")
    with open(DATA_JSON_PATH, "r") as f:
        test_data = json.load(f)

    keys = list(test_data.keys())

    # Tentukan jumlah sampel multi-data yang ingin diekstrak (misal: 5 sampel)
    num_samples = min(5, len(keys)) # Mencegah IndexError jika total data kurang dari 5

    c_code = "#ifndef TEST_SAMPLES_H\n#define TEST_SAMPLES_H\n\n"
    c_code += "// Kumpulan Sampel Pengujian Multi-Data untuk ESP32-S3\n\n"

    for i in range(num_samples):
        k = keys[i]
        sample_eq = test_data[k]["Z"][:700]          # Sinyal Gempa (700 sampel)
        sample_noise = test_data[k]["Z_noise"][-700:] # Sinyal Derau (700 sampel)

        # Format Array C untuk Gempa
        c_code += f"const float test_wave_earthquake_{i+1}[700] = {{\n    "
        c_code += ", ".join([f"{val:.6f}" for val in sample_eq])
        c_code += "\n};\n\n"

        # Format Array C untuk Derau
        c_code += f"const float test_wave_noise_{i+1}[700] = {{\n    "
        c_code += ", ".join([f"{val:.6f}" for val in sample_noise])
        c_code += "\n};\n\n"

    c_code += "#endif // TEST_SAMPLES_H"

    with open(OUTPUT_FILE, "w") as f:
        f.write(c_code)

    print(f"✅ Berhasil mengekstrak {num_samples} pasang gelombang uji ke: {OUTPUT_FILE}")

if __name__ == "__main__":
    main()